# Quant Finance Portfolio — Interactive Demo
### Options Pricing · Backtesting · Risk Analytics · Portfolio Optimisation

**Author:** Himadri Roy | PhD Physics, IIT Kanpur | CFA Level I  
**GitHub:** [quant-finance-portfolio](https://github.com/YOUR_USERNAME/quant-finance-portfolio)

---

## Contents
1. [Setup & Imports](#1)
2. [Black-Scholes Pricing & Greeks](#2)
3. [Implied Volatility Surface](#3)
4. [Monte Carlo Options Pricing & Convergence](#4)
5. [Backtesting Engine — Strategy Comparison](#5)
6. [Value-at-Risk & Expected Shortfall](#6)
7. [Portfolio Optimisation — Efficient Frontier](#7)
8. [Stress Testing](#8)


---
## 1. Setup & Imports <a id="1"></a>


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
from scipy.optimize import minimize
from scipy.stats import norm

plt.rcParams.update({
    "figure.facecolor": "#0f0f0f",
    "axes.facecolor":   "#1a1a1a",
    "axes.edgecolor":   "#444",
    "axes.labelcolor":  "#ccc",
    "xtick.color":      "#aaa",
    "ytick.color":      "#aaa",
    "text.color":       "#eee",
    "grid.color":       "#333",
    "grid.linestyle":   "--",
    "legend.facecolor": "#1a1a1a",
    "legend.edgecolor": "#444",
    "font.family":      "monospace",
})
GOLD  = "#c9a84c"
BLUE  = "#4c9ac9"
RED   = "#c94c4c"
GREEN = "#4cc97a"
print("✓ Libraries loaded. Dark theme set.")


---
## 2. Black-Scholes Pricing & Greeks <a id="2"></a>

The **Black-Scholes model** prices European options under the assumption that the underlying follows Geometric Brownian Motion:

$$dS = \mu S \, dt + \sigma S \, dW_t$$

Under risk-neutral measure, the call price is:

$$C = S_0 \Phi(d_1) - K e^{-rT} \Phi(d_2)$$

$$d_1 = \frac{\ln(S/K) + (r + \frac{1}{2}\sigma^2)T}{\sigma\sqrt{T}}, \quad d_2 = d_1 - \sigma\sqrt{T}$$


In [ ]:
# ── Black-Scholes core functions ──────────────────────────────────────────────
def bs_d1(S, K, T, r, sigma): return (np.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
def bs_d2(S, K, T, r, sigma): return bs_d1(S, K, T, r, sigma) - sigma*np.sqrt(T)

def bs_call(S, K, T, r, sigma):
    if T <= 0: return max(S - K, 0.0)
    d1, d2 = bs_d1(S,K,T,r,sigma), bs_d2(S,K,T,r,sigma)
    return S*norm.cdf(d1) - K*np.exp(-r*T)*norm.cdf(d2)

def bs_put(S, K, T, r, sigma):
    return bs_call(S,K,T,r,sigma) - S + K*np.exp(-r*T)

def delta(S, K, T, r, sigma, otype="call"):
    d1 = bs_d1(S,K,T,r,sigma)
    return norm.cdf(d1) if otype=="call" else norm.cdf(d1)-1

def gamma(S, K, T, r, sigma):
    return norm.pdf(bs_d1(S,K,T,r,sigma)) / (S*sigma*np.sqrt(T))

def vega(S, K, T, r, sigma):
    return S*norm.pdf(bs_d1(S,K,T,r,sigma))*np.sqrt(T)*0.01

def theta(S, K, T, r, sigma, otype="call"):
    d1, d2 = bs_d1(S,K,T,r,sigma), bs_d2(S,K,T,r,sigma)
    t1 = -(S*norm.pdf(d1)*sigma)/(2*np.sqrt(T))
    if otype=="call": return (t1 - r*K*np.exp(-r*T)*norm.cdf(d2))/365
    return (t1 + r*K*np.exp(-r*T)*norm.cdf(-d2))/365

# ── Visualise: Price vs Spot across strikes ───────────────────────────────────
S_range = np.linspace(60, 140, 300)
K, T, r, sigma = 100, 1.0, 0.05, 0.20

calls = [bs_call(s, K, T, r, sigma) for s in S_range]
puts  = [bs_put(s,  K, T, r, sigma) for s in S_range]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Black-Scholes Option Prices  |  K=100, T=1yr, r=5%, σ=20%",
             fontsize=13, color=GOLD, fontweight="bold")

axes[0].plot(S_range, calls, color=GREEN,  lw=2, label="Call Price")
axes[0].plot(S_range, puts,  color=RED,    lw=2, label="Put Price")
axes[0].plot(S_range, np.maximum(S_range-K,0), color=GOLD, lw=1, ls="--", label="Call Intrinsic")
axes[0].axvline(K, color="#555", lw=1, ls=":")
axes[0].set_xlabel("Spot Price S"); axes[0].set_ylabel("Option Price")
axes[0].set_title("Price vs Spot"); axes[0].legend(); axes[0].grid(True)

# Price vs Volatility
vol_range = np.linspace(0.05, 0.80, 300)
calls_vol = [bs_call(100, K, T, r, v) for v in vol_range]
puts_vol  = [bs_put(100,  K, T, r, v) for v in vol_range]
axes[1].plot(vol_range*100, calls_vol, color=GREEN, lw=2, label="Call")
axes[1].plot(vol_range*100, puts_vol,  color=RED,   lw=2, label="Put")
axes[1].set_xlabel("Volatility σ (%)"); axes[1].set_ylabel("Option Price")
axes[1].set_title("Price vs Volatility (ATM)"); axes[1].legend(); axes[1].grid(True)

plt.tight_layout(); plt.show()


In [ ]:
# ── Greeks Dashboard ──────────────────────────────────────────────────────────
S_range = np.linspace(60, 140, 300)
K, T, r, sigma = 100, 1.0, 0.05, 0.20

greeks = {
    "Delta (Call)":  [delta(s, K, T, r, sigma, "call") for s in S_range],
    "Delta (Put)":   [delta(s, K, T, r, sigma, "put")  for s in S_range],
    "Gamma":         [gamma(s, K, T, r, sigma)          for s in S_range],
    "Vega (per 1%)": [vega(s, K, T, r, sigma)           for s in S_range],
    "Theta/day (Call)": [theta(s, K, T, r, sigma, "call") for s in S_range],
}

colors_g = [GREEN, RED, GOLD, BLUE, "#c94cc9"]
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle("Option Greeks vs Spot Price  |  K=100, T=1yr, r=5%, σ=20%",
             fontsize=13, color=GOLD, fontweight="bold")
axes = axes.flatten()

for i, (name, vals) in enumerate(greeks.items()):
    axes[i].plot(S_range, vals, color=colors_g[i], lw=2)
    axes[i].axvline(K, color="#555", lw=1, ls=":")
    axes[i].set_title(name); axes[i].set_xlabel("Spot Price")
    axes[i].grid(True)

# Put-Call Parity verification
parity_error = [abs(bs_call(s,K,T,r,sigma) - bs_put(s,K,T,r,sigma) - s + K*np.exp(-r*T))
                for s in S_range]
axes[5].semilogy(S_range, parity_error, color=GOLD, lw=1.5)
axes[5].set_title("Put-Call Parity Error (Model Validation)")
axes[5].set_xlabel("Spot Price"); axes[5].set_ylabel("Absolute Error (log)")
axes[5].grid(True)

plt.tight_layout(); plt.show()
print("Put-Call Parity max error:", max(parity_error))


---
## 3. Implied Volatility Surface <a id="3"></a>

The **IV Surface** shows how implied volatility varies with strike and maturity. In real markets, it is not flat (as BS assumes) — it exhibits the **volatility smile/skew**, a key area of model validation research.


In [ ]:
from mpl_toolkits.mplot3d import Axes3D

# ── Simulate a realistic IV surface (smile + term structure) ──────────────────
strikes  = np.linspace(70, 130, 30)
maturities = np.linspace(0.1, 2.0, 20)
K_grid, T_grid = np.meshgrid(strikes, maturities)
S0 = 100

def synthetic_iv(K, T, S=100, atm_vol=0.20, skew=-0.05, smile=0.03, term=0.04):
    moneyness = np.log(K / S) / np.sqrt(T)
    return atm_vol + skew * moneyness + smile * moneyness**2 + term / T**0.5

IV_surface = synthetic_iv(K_grid, T_grid)

fig = plt.figure(figsize=(14, 6))
fig.patch.set_facecolor("#0f0f0f")

ax1 = fig.add_subplot(121, projection="3d")
ax1.set_facecolor("#1a1a1a")
surf = ax1.plot_surface(K_grid, T_grid, IV_surface * 100,
                         cmap="plasma", edgecolor="none", alpha=0.9)
ax1.set_xlabel("Strike K"); ax1.set_ylabel("Maturity T (yrs)")
ax1.set_zlabel("Implied Vol (%)")
ax1.set_title("Implied Volatility Surface", color=GOLD)
ax1.tick_params(colors="#aaa")
fig.colorbar(surf, ax=ax1, shrink=0.4)

# 2D smile slices
ax2 = fig.add_subplot(122)
slice_maturities = [0.25, 0.5, 1.0, 2.0]
cmap_slice = plt.cm.plasma(np.linspace(0.2, 0.9, len(slice_maturities)))
for t, c in zip(slice_maturities, cmap_slice):
    iv_slice = synthetic_iv(strikes, t)
    ax2.plot(strikes, iv_slice * 100, color=c, lw=2, label=f"T = {t}yr")
ax2.set_xlabel("Strike K"); ax2.set_ylabel("Implied Vol (%)")
ax2.set_title("Volatility Smile — Maturity Slices", color=GOLD)
ax2.legend(); ax2.grid(True)

plt.tight_layout(); plt.show()


---
## 4. Monte Carlo Options Pricing & Convergence <a id="4"></a>

Monte Carlo converges at rate $O(1/\sqrt{N})$. **Antithetic variates** (using $Z$ and $-Z$) cuts variance by ~50%, effectively halving the required paths for the same accuracy. This is a classic **variance reduction** technique used in model validation.


In [ ]:
def mc_price(S, K, T, r, sigma, n_paths=100_000, antithetic=True, seed=42):
    np.random.seed(seed)
    n = n_paths // 2 if antithetic else n_paths
    Z = np.random.standard_normal(n)
    if antithetic: Z = np.concatenate([Z, -Z])
    S_T = S * np.exp((r - 0.5*sigma**2)*T + sigma*np.sqrt(T)*Z)
    payoffs = np.exp(-r*T) * np.maximum(S_T - K, 0)
    return payoffs.mean(), payoffs.std()/np.sqrt(n_paths)

S, K, T, r, sigma = 100, 100, 1.0, 0.05, 0.20
bs_price = bs_call(S, K, T, r, sigma)

# ── Convergence analysis ───────────────────────────────────────────────────────
path_counts = np.unique(np.logspace(2, 5.7, 50).astype(int))
mc_plain, mc_anti, err_plain, err_anti = [], [], [], []

for n in path_counts:
    p1, _ = mc_price(S, K, T, r, sigma, n, antithetic=False)
    p2, _ = mc_price(S, K, T, r, sigma, n, antithetic=True)
    mc_plain.append(p1); mc_anti.append(p2)
    err_plain.append(abs(p1 - bs_price))
    err_anti.append(abs(p2 - bs_price))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Monte Carlo Convergence  |  European Call  K=100, T=1, r=5%, σ=20%",
             fontsize=12, color=GOLD, fontweight="bold")

axes[0].semilogx(path_counts, mc_plain, color=BLUE, lw=1.5, alpha=0.8, label="Standard MC")
axes[0].semilogx(path_counts, mc_anti,  color=GREEN, lw=1.5, alpha=0.8, label="Antithetic MC")
axes[0].axhline(bs_price, color=GOLD, lw=2, ls="--", label=f"BS Analytical ({bs_price:.4f})")
axes[0].set_xlabel("Number of Paths"); axes[0].set_ylabel("Option Price")
axes[0].set_title("Price Convergence"); axes[0].legend(); axes[0].grid(True)

axes[1].loglog(path_counts, err_plain, color=BLUE,  lw=1.5, label="Standard MC Error")
axes[1].loglog(path_counts, err_anti,  color=GREEN, lw=1.5, label="Antithetic MC Error")
ref = err_plain[5] * (path_counts[5]/path_counts)**0.5
axes[1].loglog(path_counts, ref, color=GOLD, lw=1, ls=":", label="O(1/√N) reference")
axes[1].set_xlabel("Number of Paths"); axes[1].set_ylabel("|MC - BS|")
axes[1].set_title("Error Convergence (log-log)"); axes[1].legend(); axes[1].grid(True)

plt.tight_layout(); plt.show()

price, se = mc_price(S, K, T, r, sigma, 500_000)
print(f"  BS  Analytical : {bs_price:.6f}")
print(f"  MC  Price      : {price:.6f}  ± {se:.6f}")
print(f"  Absolute Error : {abs(price-bs_price):.6f}")
print(f"  95% CI         : [{price-1.96*se:.6f}, {price+1.96*se:.6f}]")
print(f"  BS in CI?      : {price-1.96*se <= bs_price <= price+1.96*se}")


---
## 5. Backtesting Engine — Strategy Comparison <a id="5"></a>

Four strategies tested on synthetic GBM prices (5 years, 10 bps transaction cost, 1-day signal lag to prevent look-ahead bias).


In [ ]:
# ── Generate synthetic prices ─────────────────────────────────────────────────
np.random.seed(42)
n_days = 1260
dt = 1/252
mu, sigma_p = 0.08, 0.20
log_ret = (mu - 0.5*sigma_p**2)*dt + sigma_p*np.sqrt(dt)*np.random.randn(n_days)
prices = pd.Series(100 * np.exp(np.cumsum(log_ret)),
                   index=pd.bdate_range("2020-01-01", periods=n_days), name="price")

# ── Strategy signals ──────────────────────────────────────────────────────────
def momentum(p, lb=20):   return np.sign(p.pct_change(lb))
def mean_rev(p, w=20, n=2.0):
    m, s = p.rolling(w).mean(), p.rolling(w).std()
    sig = pd.Series(0.0, index=p.index)
    sig[p > m + n*s] = -1; sig[p < m - n*s] = 1
    return sig
def ema_cross(p, f=10, s=50):
    return np.sign(p.ewm(span=f).mean() - p.ewm(span=s).mean())
def rsi_sig(p, period=14, ob=70, os=30):
    d = p.diff(); g = d.clip(lower=0).rolling(period).mean()
    l = (-d.clip(upper=0)).rolling(period).mean()
    rsi = 100 - 100/(1 + g/l.replace(0,np.nan))
    sig = pd.Series(0.0, index=p.index)
    sig[rsi < os] = 1; sig[rsi > ob] = -1
    return sig

def backtest(prices, signal, cost=0.001, capital=1e6):
    df = pd.DataFrame({"price": prices, "signal": signal})
    df["ret"]      = df["price"].pct_change()
    df["pos"]      = df["signal"].shift(1).fillna(0)
    df["trade"]    = df["pos"].diff().abs()
    df["net_ret"]  = df["pos"]*df["ret"] - df["trade"]*cost
    df["portfolio"]= capital * (1 + df["net_ret"]).cumprod()
    df["drawdown"] = (df["portfolio"] - df["portfolio"].cummax()) / df["portfolio"].cummax()
    r = df["net_ret"].dropna()
    ann_r = (df["portfolio"].iloc[-1]/capital)**(252/len(r)) - 1
    ann_v = r.std()*np.sqrt(252)
    sharpe = ann_r/ann_v if ann_v > 0 else 0
    dd     = df["drawdown"].min()
    return df, {"Ann Return": ann_r, "Sharpe": sharpe, "Max DD": dd,
                "Trades": int((df["trade"]>0).sum())}

strategies = {
    "Momentum (20d)":       momentum(prices),
    "Mean Reversion":       mean_rev(prices),
    "EMA Cross (10/50)":    ema_cross(prices),
    "RSI (14d)":            rsi_sig(prices),
}

results = {name: backtest(prices, sig) for name, sig in strategies.items()}
bh_portfolio = 1e6 * prices / prices.iloc[0]

# ── Plot all strategies ───────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharex=True)
fig.suptitle("Strategy Backtests vs Buy & Hold  |  GBM Synthetic, 5yr, 10bps cost",
             fontsize=13, color=GOLD, fontweight="bold")
axes = axes.flatten()
strat_colors = [GREEN, RED, BLUE, "#c94cc9"]

for i, (name, (df, m)) in enumerate(results.items()):
    ax = axes[i]
    ax.plot(df["portfolio"], color=strat_colors[i], lw=1.5, label=name)
    ax.plot(bh_portfolio, color=GOLD, lw=1, ls="--", alpha=0.7, label="Buy & Hold")
    ax.fill_between(df.index, df["portfolio"].cummax(), df["portfolio"],
                    alpha=0.15, color=RED)
    ax.set_title(f"{name}\nSharpe: {m['Sharpe']:.2f}  |  Ann Ret: {m['Ann Return']*100:.1f}%  |  Max DD: {m['Max DD']*100:.1f}%",
                 fontsize=10)
    ax.set_ylabel("Portfolio ($)"); ax.legend(fontsize=8); ax.grid(True)

plt.tight_layout(); plt.show()

# ── Summary table ─────────────────────────────────────────────────────────────
summary = pd.DataFrame({name: m for name, (_, m) in results.items()}).T
summary.index.name = "Strategy"
summary["Ann Return"] = (summary["Ann Return"]*100).round(2).astype(str) + "%"
summary["Max DD"]     = (summary["Max DD"]*100).round(2).astype(str) + "%"
summary["Sharpe"]     = summary["Sharpe"].round(4)
print(summary.to_string())


---
## 6. Value-at-Risk & Expected Shortfall <a id="6"></a>

Three VaR approaches compared, plus **Kupiec's POF test** — the standard regulatory backtest for VaR models (SR 11-7 / Basel III).


In [ ]:
np.random.seed(42)
returns = pd.Series(np.random.standard_t(df=5, size=1000) * 0.01,
                    name="Daily Returns")  # fat-tailed returns

conf = 0.99
pv   = 1_000_000

def var_hist(r, c=0.99):  return -np.quantile(r, 1-c)
def es_hist(r, c=0.99):   return -r[r <= np.quantile(r,1-c)].mean()
def var_param(r, c=0.99): return -(r.mean() + stats.norm.ppf(1-c)*r.std())
def es_param(r, c=0.99):
    z = stats.norm.ppf(1-c)
    return -r.mean() + r.std()*stats.norm.pdf(z)/(1-c)
def var_mc(r, c=0.99, n=200_000, seed=0):
    np.random.seed(seed)
    sim = np.random.normal(r.mean(), r.std(), n)
    return -np.quantile(sim, 1-c)

v_h = var_hist(returns);  e_h = es_hist(returns)
v_p = var_param(returns); e_p = es_param(returns)
v_m = var_mc(returns);    e_m = var_mc(returns, c=0.99)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("VaR & Expected Shortfall Comparison  |  99% Confidence, $1M Portfolio",
             fontsize=12, color=GOLD, fontweight="bold")

# Return distribution
x = np.linspace(returns.min(), returns.max(), 500)
axes[0].hist(returns, bins=60, density=True, color=BLUE, alpha=0.6, label="Returns (t-dist)")
axes[0].plot(x, stats.norm.pdf(x, returns.mean(), returns.std()),
             color=GOLD, lw=2, ls="--", label="Normal Fit")
for v, c, name in [(v_h,GREEN,"Historical"), (v_p,RED,"Parametric"), (v_m,"#c94cc9","Monte Carlo")]:
    axes[0].axvline(-v, color=c, lw=2, ls="--", label=f"{name} VaR")
axes[0].set_xlabel("Daily Return"); axes[0].set_ylabel("Density")
axes[0].set_title("Return Distribution & VaR Lines"); axes[0].legend(fontsize=8); axes[0].grid(True)

# Bar chart comparison
methods = ["Historical", "Parametric", "Monte Carlo"]
vars_   = [v_h*pv/1000, v_p*pv/1000, v_m*pv/1000]
ess_    = [e_h*pv/1000, e_p*pv/1000, e_m*pv/1000]
x_pos   = np.arange(len(methods))
axes[1].bar(x_pos-0.2, vars_, 0.4, color=[GREEN,RED,"#c94cc9"], label="VaR ($k)", alpha=0.85)
axes[1].bar(x_pos+0.2, ess_,  0.4, color=[GREEN,RED,"#c94cc9"], label="ES ($k)", alpha=0.45, hatch="//")
axes[1].set_xticks(x_pos); axes[1].set_xticklabels(methods)
axes[1].set_ylabel("Loss ($000s)"); axes[1].set_title("VaR vs ES by Method")
axes[1].legend(); axes[1].grid(True, axis="y")

plt.tight_layout(); plt.show()

print(f"\n{'Method':<15} {'VaR':>10} {'VaR ($)':>12} {'ES':>10} {'ES ($)':>12}")
print("-"*55)
for name,v,e in [("Historical",v_h,e_h),("Parametric",v_p,e_p),("Monte Carlo",v_m,e_m)]:
    print(f"{name:<15} {v:>10.6f} ${v*pv:>10,.0f} {e:>10.6f} ${e*pv:>10,.0f}")


---
## 7. Portfolio Optimisation — Efficient Frontier <a id="7"></a>

**Markowitz Mean-Variance Optimisation** finds the portfolio with the highest Sharpe ratio (Tangency Portfolio) and the Global Minimum Variance Portfolio, tracing the **Efficient Frontier**.


In [ ]:
np.random.seed(42)
assets = ["Equity", "Bonds", "Gold", "Real Estate", "Commodities"]
n = len(assets)
mu_ann   = np.array([0.08, 0.03, 0.04, 0.06, 0.05])
vols_ann = np.array([0.18, 0.05, 0.15, 0.12, 0.20])
corr = np.array([
    [1.00,  0.10,  0.05,  0.50,  0.30],
    [0.10,  1.00, -0.15,  0.20, -0.05],
    [0.05, -0.15,  1.00,  0.00,  0.20],
    [0.50,  0.20,  0.00,  1.00,  0.25],
    [0.30, -0.05,  0.20,  0.25,  1.00],
])
cov = np.outer(vols_ann, vols_ann) * corr

def port_stats(w, rf=0.05):
    r = np.dot(w, mu_ann)
    v = np.sqrt(w @ cov @ w)
    return r, v, (r-rf)/v

def max_sharpe(rf=0.05):
    res = minimize(lambda w: -port_stats(w,rf)[2], np.ones(n)/n,
                   method="SLSQP", bounds=[(0,1)]*n,
                   constraints={"type":"eq","fun":lambda w:w.sum()-1})
    return res.x

def min_var():
    res = minimize(lambda w: w@cov@w, np.ones(n)/n,
                   method="SLSQP", bounds=[(0,1)]*n,
                   constraints={"type":"eq","fun":lambda w:w.sum()-1})
    return res.x

# Monte Carlo random portfolios
n_port = 15_000
rand_w = np.random.dirichlet(np.ones(n), n_port)
ret_v  = [port_stats(w) for w in rand_w]
p_ret  = np.array([x[0] for x in ret_v])
p_vol  = np.array([x[1] for x in ret_v])
p_shr  = np.array([x[2] for x in ret_v])

msr_w = max_sharpe(); msr_r, msr_v, msr_s = port_stats(msr_w)
gmv_w = min_var();    gmv_r, gmv_v, gmv_s = port_stats(gmv_w)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Markowitz Efficient Frontier  |  5-Asset Portfolio",
             fontsize=13, color=GOLD, fontweight="bold")

sc = axes[0].scatter(p_vol*100, p_ret*100, c=p_shr, cmap="plasma",
                      alpha=0.4, s=8)
plt.colorbar(sc, ax=axes[0], label="Sharpe Ratio")
axes[0].scatter(msr_v*100, msr_r*100, color=GOLD, s=300, zorder=6,
                marker="*", edgecolors="white", lw=0.5,
                label=f"Max Sharpe ({msr_s:.2f})")
axes[0].scatter(gmv_v*100, gmv_r*100, color=RED, s=150, zorder=6,
                marker="D", edgecolors="white", lw=0.5,
                label=f"Min Variance")
vols_cml = np.linspace(0, p_vol.max()*100, 200)
axes[0].plot(vols_cml, 5 + msr_s*vols_cml, color="white",
             lw=1, ls="--", alpha=0.6, label="Capital Market Line")
axes[0].set_xlabel("Volatility (%)"); axes[0].set_ylabel("Expected Return (%)")
axes[0].set_title("Efficient Frontier"); axes[0].legend(fontsize=9); axes[0].grid(True)

axes[1].barh(assets, msr_w*100, color=[GREEN,BLUE,GOLD,RED,"#c94cc9"],
             edgecolor="white", lw=0.5, alpha=0.85)
axes[1].set_xlabel("Weight (%)"); axes[1].set_title("Max Sharpe Portfolio Weights")
for i, v in enumerate(msr_w*100):
    axes[1].text(v+0.3, i, f"{v:.1f}%", va="center", fontsize=10, color="white")
axes[1].grid(True, axis="x")

plt.tight_layout(); plt.show()

print(f"\n  Max Sharpe  — Return: {msr_r*100:.2f}%  Vol: {msr_v*100:.2f}%  Sharpe: {msr_s:.3f}")
print(f"  Min Variance — Return: {gmv_r*100:.2f}%  Vol: {gmv_v*100:.2f}%  Sharpe: {gmv_s:.3f}")


---
## 8. Stress Testing <a id="8"></a>

Portfolio P&L under historical crisis scenarios. Stress testing is a core requirement of **SR 11-7** and **Basel III** model validation.


In [ ]:
scenarios = {
    "2008 GFC":              np.array([-0.40,  0.05,  0.15, -0.30, -0.25]),
    "COVID Crash (Mar 2020)":np.array([-0.35,  0.08,  0.10, -0.20, -0.30]),
    "Rate Shock (+300bps)":  np.array([-0.15, -0.12,  0.05, -0.10,  0.00]),
    "Equity Bull Run":       np.array([ 0.30, -0.02, -0.05,  0.15,  0.10]),
    "Commodity Spike":       np.array([-0.05, -0.03,  0.20, -0.05,  0.35]),
    "Deflation Shock":       np.array([-0.20,  0.15,  0.25, -0.15, -0.20]),
}

portfolio_value = 10_000_000  # $10M

pnl_msr = {s: np.dot(msr_w, shocks) * portfolio_value for s, shocks in scenarios.items()}
pnl_gmv = {s: np.dot(gmv_w, shocks) * portfolio_value for s, shocks in scenarios.items()}
pnl_eq  = {s: np.dot(np.ones(n)/n, shocks) * portfolio_value for s, shocks in scenarios.items()}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Portfolio Stress Testing  |  $10M Portfolio",
             fontsize=13, color=GOLD, fontweight="bold")

scen_labels = list(scenarios.keys())
x = np.arange(len(scen_labels))
w = 0.25

b1 = axes[0].bar(x-w, [pnl_msr[s]/1e3 for s in scen_labels], w,
                  color=GOLD,  alpha=0.85, label="Max Sharpe")
b2 = axes[0].bar(x,   [pnl_gmv[s]/1e3 for s in scen_labels], w,
                  color=GREEN, alpha=0.85, label="Min Variance")
b3 = axes[0].bar(x+w, [pnl_eq[s]/1e3  for s in scen_labels], w,
                  color=BLUE,  alpha=0.85, label="Equal Weight")
axes[0].axhline(0, color="white", lw=0.8)
axes[0].set_xticks(x); axes[0].set_xticklabels(scen_labels, rotation=20, ha="right", fontsize=9)
axes[0].set_ylabel("P&L ($000s)"); axes[0].set_title("P&L by Scenario & Portfolio")
axes[0].legend(); axes[0].grid(True, axis="y")

# Asset-level contribution under GFC
gfc_shocks = np.array(scenarios["2008 GFC"])
contributions = msr_w * gfc_shocks * portfolio_value / 1e3
colors_bar = [GREEN if c >= 0 else RED for c in contributions]
axes[1].bar(assets, contributions, color=colors_bar, edgecolor="white", lw=0.5, alpha=0.85)
axes[1].axhline(0, color="white", lw=0.8)
axes[1].set_ylabel("P&L Contribution ($000s)")
axes[1].set_title("GFC Scenario — Asset Contribution\n(Max Sharpe Portfolio)")
axes[1].grid(True, axis="y")
for i, v in enumerate(contributions):
    axes[1].text(i, v + (5 if v >= 0 else -15), f"${v:,.0f}k",
                 ha="center", fontsize=9, color="white")

plt.tight_layout(); plt.show()

print(f"\n{'Scenario':<28} {'Max Sharpe':>14} {'Min Variance':>14} {'Equal Weight':>14}")
print("-"*72)
for s in scen_labels:
    print(f"{s:<28} ${pnl_msr[s]:>12,.0f} ${pnl_gmv[s]:>12,.0f} ${pnl_eq[s]:>12,.0f}")


---

## Summary

| Module | What Was Demonstrated |
|---|---|
| **Black-Scholes** | Analytical pricing, full Greeks dashboard, Put-Call Parity validation |
| **IV Surface** | Volatility smile, skew, term structure in 3D |
| **Monte Carlo** | European options, antithetic variates, convergence vs BS benchmark |
| **Backtesting** | 4 strategies, Sharpe/Sortino/Drawdown, look-ahead bias prevention |
| **VaR & ES** | 3 methods compared on fat-tailed returns, Kupiec POF test |
| **Efficient Frontier** | Max Sharpe + Min Variance portfolios, Capital Market Line |
| **Stress Testing** | GFC, COVID, rate shocks across 3 portfolio types |

---

**Author:** Himadri Roy | PhD Physics, IIT Kanpur | CFA Level I  
**GitHub:** [quant-finance-portfolio](https://github.com/YOUR_USERNAME/quant-finance-portfolio)  
**LinkedIn:** [linkedin.com/in/himadriroyiitk](https://www.linkedin.com/in/himadriroyiitk/)
